# V4 Multi-Class Fall Detection Pipeline

**Evidence-driven design based on V2/V3 experiments:**

| Approach | Detection | False Alerts | Speed |
|----------|-----------|--------------|-------|
| V3 Pose-CNN | 96% (53/55) | 57 total | Slow |
| V2 Multi-class | 80% (44/55) | 26 total | 25 FPS |
| V3 YOLOv8n (2-class) | 27% (15/55) | 1 | Fast |

**Decision:** Improve V2's multi-class approach with more training data.

## Architecture
- **4 Classes:** standing, bending, falling, fallen
- **Why:** `falling` captures the ACTION (transition), not just static state
- **Pipeline:** YOLOv8n (4-class) → BoTSORT tracking → Transition detection → Alert

## Target
- Detection rate: >90% (vs V2's 80%)
- False alerts: <20 total (vs V2's 26)
- Speed: >25 FPS

## Section 1: Environment Setup

In [1]:
# ============================================================================
# TEST EXISTING V2 MULTI-CLASS MODEL
# ============================================================================
# Before training V4, let's establish baseline with the existing V2 model

import os, json, cv2, torch, numpy as np
from pathlib import Path
from collections import defaultdict, deque
from ultralytics import YOLO

PROJECT_DIR = Path("/home/zmey1/VSCODE_FILES/prodesyn")
EXTRAS_DIR = PROJECT_DIR / "extras"
RESULTS_DIR = PROJECT_DIR / "results" / "v4_multiclass"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# V2 trained model
V2_MODEL = EXTRAS_DIR / "runs" / "detect" / "fall_detector_v2" / "multiclass-4" / "weights" / "best.pt"

if V2_MODEL.exists():
    print(f"V2 model found: {V2_MODEL}")
    print(f"Size: {V2_MODEL.stat().st_size / 1024 / 1024:.2f} MB")
    
    # Load and check classes
    model = YOLO(str(V2_MODEL))
    print(f"Classes: {model.names}")
else:
    print(f"V2 model not found: {V2_MODEL}")

V2 model found: /home/zmey1/VSCODE_FILES/prodesyn/extras/runs/detect/fall_detector_v2/multiclass-4/weights/best.pt
Size: 5.94 MB
Classes: {0: 'Fall-Detected'}


In [2]:
# Load existing V2 results and compute metrics
V2_RESULTS_PATH = EXTRAS_DIR / "results" / "v2_comparison" / "v2_event_results.json"
V2_MODEL_PATH = EXTRAS_DIR / "results" / "v2_comparison" / "best.pt"

# Load manifest
with open(PROJECT_DIR / "shared_val_manifest.json") as f:
    manifest = json.load(f)
gt_lookup = {v["video_uid"]: v for v in manifest["videos"]}

# Load V2 results
with open(V2_RESULTS_PATH) as f:
    v2_results = json.load(f)

print(f"V2 results: {len(v2_results)} videos")

# Compute metrics
detected = 0
missed = 0
fa_vids = 0
total_fas = 0
delays = []

for video_uid, res in v2_results.items():
    gt = gt_lookup.get(video_uid, {})
    has_fall = gt.get("has_fall", False)
    
    if has_fall:
        if res.get("fall_detected", False):
            detected += 1
            if res.get("detection_delay_frames") is not None:
                delays.append(res["detection_delay_frames"])
        else:
            missed += 1
    else:
        if res.get("fall_detected", False):
            fa_vids += 1
            total_fas += res.get("num_false_alerts", 1)

total_fall = detected + missed
med_delay = np.median(delays) if delays else float("nan")

print(f"\n{'='*60}")
print(f"V2 BASELINE (from saved results)")
print(f"{'='*60}")
print(f"Detection: {detected}/{total_fall} ({100*detected/total_fall:.1f}%)")
print(f"Missed: {missed}")
print(f"FA videos: {fa_vids}")
print(f"Total FAs: {total_fas}")
print(f"Median delay: {med_delay:.1f} frames")
print(f"{'='*60}")

V2 results: 86 videos

V2 BASELINE (from saved results)
Detection: 44/55 (80.0%)
Missed: 11
FA videos: 19
Total FAs: 26
Median delay: 18.0 frames


In [3]:
# Check what model V2 actually uses
v2_model = YOLO(str(V2_MODEL_PATH))
print(f"V2 Model classes: {v2_model.names}")
print(f"Number of classes: {len(v2_model.names)}")

# This is a single-class model (Fall-Detected only)
# The V2 pipeline must use pose + this detector in a specific way
# Let me check how V2 actually works by looking at the local_v2.ipynb pipeline

V2 Model classes: {0: 'Fall-Detected'}
Number of classes: 1


In [4]:
# Implement V2's actual pipeline (pose + single-class detector + IoU fusion)

def compute_iou(box1, box2):
    """Compute IoU between two boxes [x1,y1,x2,y2]"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter = max(0, x2-x1) * max(0, y2-y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    
    return inter / union if union > 0 else 0


class DetectorFallPipeline:
    """
    V2 Pipeline: YOLOv8n-pose (tracking) + single-class fall detector + IoU fusion.
    
    Logic:
    - Track persons with pose model
    - Detect "Fall-Detected" regions with fall detector
    - If fall box overlaps person (IoU > thresh) → "fallen"
    - If no overlap → "normal" (person is upright)
    """
    
    def __init__(self, fall_model_path, pose_model_path="yolov8n-pose.pt",
                 confirm_frames=15, fallen_iou_thresh=0.3, history_len=60):
        self.fall_model = YOLO(str(fall_model_path))
        self.pose_model = YOLO(str(pose_model_path))
        self.confirm_frames = confirm_frames
        self.fallen_iou_thresh = fallen_iou_thresh
        self.history_len = history_len
        self.reset()
    
    def reset(self):
        self._history = defaultdict(lambda: deque(maxlen=self.history_len))
        self._alerted = defaultdict(bool)
        self._events = []
        self.pose_model.predictor = None
    
    def process_frame(self, frame, frame_idx):
        events = []
        
        # Stage 1: Pose tracking for stable person boxes
        pose_res = self.pose_model.track(
            frame, persist=True, tracker="botsort.yaml",
            conf=0.25, iou=0.45, verbose=False
        )
        
        person_boxes = []
        if pose_res and pose_res[0].boxes is not None and pose_res[0].boxes.id is not None:
            ids = pose_res[0].boxes.id.cpu().numpy().astype(int)
            xyxys = pose_res[0].boxes.xyxy.cpu().numpy()
            for tid, xyxy in zip(ids, xyxys):
                person_boxes.append((int(tid), xyxy.tolist()))
        
        # Stage 2: Fall detector (single-class)
        det_res = self.fall_model(frame, conf=0.25, verbose=False)
        
        fall_boxes = []
        if det_res and det_res[0].boxes is not None:
            for xyxy in det_res[0].boxes.xyxy.cpu().numpy():
                fall_boxes.append(xyxy.tolist())
        
        # Stage 3: IoU fusion
        for tid, p_box in person_boxes:
            # Check if any fall box overlaps this person
            is_fallen = False
            for f_box in fall_boxes:
                if compute_iou(p_box, f_box) > self.fallen_iou_thresh:
                    is_fallen = True
                    break
            
            label = "fallen" if is_fallen else "normal"
            self._history[tid].append(label)
            
            # Check alert conditions
            if self._alerted[tid]:
                # Recovery check
                hist = list(self._history[tid])
                normal_count = sum(1 for h in hist[-20:] if h == "normal")
                if normal_count >= 15:
                    self._alerted[tid] = False
                continue
            
            # Fall detection
            hist = list(self._history[tid])
            fallen_tail = hist[-self.confirm_frames:]
            
            if len(fallen_tail) == self.confirm_frames and all(h == "fallen" for h in fallen_tail):
                had_normal = any(h == "normal" for h in hist[:-self.confirm_frames])
                if had_normal:
                    self._alerted[tid] = True
                    events.append({
                        "frame_idx": frame_idx,
                        "track_id": int(tid),
                        "event_type": "fall_confirmed"
                    })
                    self._events.append(events[-1])
        
        return events
    
    def run_on_video(self, video_path):
        self.reset()
        cap = cv2.VideoCapture(str(video_path))
        all_events = []
        frame_idx = 0
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            events = self.process_frame(frame, frame_idx)
            all_events.extend(events)
            frame_idx += 1
        
        cap.release()
        return all_events


print("DetectorFallPipeline (V2 style) defined.")

DetectorFallPipeline (V2 style) defined.


In [ ]:
# Test V2 pipeline on validation videos
# Helper to resolve video paths
def resolve_video_path(video_path):
    p = Path(video_path)
    if p.exists():
        return str(p)
    try:
        rel = p.relative_to(PROJECT_DIR)
        extras_path = EXTRAS_DIR / rel
        if extras_path.exists():
            return str(extras_path)
    except ValueError:
        pass
    # Try direct extras path
    for parent in [EXTRAS_DIR, EXTRAS_DIR / "archive", EXTRAS_DIR / "archive_ur"]:
        test_path = parent / p.name
        if test_path.exists():
            return str(test_path)
    return None

# Run V2 pipeline
print("Running V2-style pipeline on validation videos...")
v2_pipeline = DetectorFallPipeline(V2_MODEL_PATH, confirm_frames=15)

v2_test_results = {}
for i, vid in enumerate(manifest["videos"]):
    video_path = resolve_video_path(vid["video_path"])
    if video_path is None:
        continue
    
    events = v2_pipeline.run_on_video(video_path)
    v2_test_results[vid["video_uid"]] = {
        "fall_detected": len(events) > 0,
        "events": events,
        "num_events": len(events)
    }
    
    if (i + 1) % 20 == 0:
        print(f"  Processed {i+1}/{len(manifest['videos'])} videos")

print(f"\nProcessed {len(v2_test_results)} videos")

Running V2-style pipeline on validation videos...
  Processed 20/86 videos
  Processed 40/86 videos


In [ ]:
import os, sys, json, shutil, warnings, cv2, yaml, torch
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict, deque, Counter
from ultralytics import YOLO
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Detect environment
IS_KAGGLE = os.path.exists("/kaggle")

if IS_KAGGLE:
    WORKING_DIR = Path("/kaggle/working")
    INPUT_DIR = Path("/kaggle/input")
    RESULTS_DIR = WORKING_DIR / "results"
    PROJECT_DIR = WORKING_DIR
    EXTRAS_DIR = INPUT_DIR  # Datasets uploaded as Kaggle input
else:
    PROJECT_DIR = Path("/home/zmey1/VSCODE_FILES/prodesyn")
    EXTRAS_DIR = PROJECT_DIR / "extras"
    RESULTS_DIR = PROJECT_DIR / "results" / "v4_multiclass"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# GPU check
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")
    DEVICE = "cuda"
else:
    print("WARNING: No GPU detected!")
    DEVICE = "cpu"

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Results: {RESULTS_DIR}")

## Section 2: Download ENME472 4-Class Dataset

The ENME472 dataset has 4 classes: **standing, bending, falling, fallen**

This is the key difference from V3 YOLOv8n (which only had 2 classes and failed).

In [ ]:
# Download ENME472 multi-class dataset from Roboflow
# Classes: bending (0), fallen (1), falling (2), standing (3)

DATASET_DIR = PROJECT_DIR / "Fall-Detection-v4-multiclass"

if not IS_KAGGLE:
    # Install roboflow if needed
    try:
        from roboflow import Roboflow
    except ImportError:
        %pip install roboflow -q
        from roboflow import Roboflow
    
    # Download ENME472 dataset (4 classes)
    if not DATASET_DIR.exists():
        print("Downloading ENME472 4-class dataset...")
        rf = Roboflow(api_key="YOUR_API_KEY")  # Replace or use environment variable
        project = rf.workspace("enme472-bgu7z").project("fall-detection-67grz")
        dataset = project.version(2).download("yolov8", location=str(DATASET_DIR))
        print(f"Downloaded to: {DATASET_DIR}")
    else:
        print(f"Dataset exists: {DATASET_DIR}")
else:
    # On Kaggle, dataset should be uploaded as input
    print("Kaggle mode: Looking for uploaded dataset...")
    possible_paths = list(INPUT_DIR.rglob("data.yaml"))
    if possible_paths:
        DATASET_DIR = possible_paths[0].parent
        print(f"Found dataset: {DATASET_DIR}")
    else:
        print("ERROR: Upload Fall-Detection-v4-multiclass to Kaggle input")

In [ ]:
# Verify dataset and check classes
DATA_YAML = DATASET_DIR / "data.yaml"

if DATA_YAML.exists():
    with open(DATA_YAML) as f:
        data_cfg = yaml.safe_load(f)
    
    CLASS_NAMES = data_cfg.get("names", [])
    print(f"Classes: {CLASS_NAMES}")
    print(f"NC: {data_cfg.get('nc', len(CLASS_NAMES))}")
    
    # Count images per split
    for split in ["train", "valid", "test"]:
        img_dir = DATASET_DIR / split / "images"
        if img_dir.exists():
            count = len(list(img_dir.glob("*")))
            print(f"  {split}: {count} images")
    
    # Class distribution in training data
    lbl_dir = DATASET_DIR / "train" / "labels"
    if lbl_dir.exists():
        class_counts = Counter()
        for lbl_file in lbl_dir.glob("*.txt"):
            for line in lbl_file.read_text().strip().splitlines():
                if line.strip():
                    cls_id = int(line.split()[0])
                    class_counts[cls_id] += 1
        
        print(f"\nClass distribution (train):")
        for cls_id in sorted(class_counts.keys()):
            name = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)
            print(f"  [{cls_id}] {name}: {class_counts[cls_id]}")
else:
    print(f"ERROR: {DATA_YAML} not found")
    CLASS_NAMES = []

## Section 3: Data Augmentation (Optional)

If ENME472 is small (<2000 images), augment with:
- Fall-Detection-1 (map to `fallen`)
- UR Fall sequences (auto-label with pose)

In [ ]:
# Check if we need augmentation
train_imgs = list((DATASET_DIR / "train" / "images").glob("*")) if (DATASET_DIR / "train" / "images").exists() else []
print(f"Current training images: {len(train_imgs)}")

NEED_AUGMENTATION = len(train_imgs) < 2000
print(f"Need augmentation: {NEED_AUGMENTATION}")

In [ ]:
# Augment with Fall-Detection-1 if needed (map single class to 'fallen')
if NEED_AUGMENTATION and not IS_KAGGLE:
    FD1_DIR = EXTRAS_DIR / "Fall-Detection-1"
    
    if FD1_DIR.exists():
        # Find the 'fallen' class index in our dataset
        fallen_idx = None
        for i, name in enumerate(CLASS_NAMES):
            if name.lower() in ["fallen", "fall-detected", "fall"]:
                fallen_idx = i
                break
        
        if fallen_idx is not None:
            print(f"Augmenting with Fall-Detection-1 (mapping to class {fallen_idx}: {CLASS_NAMES[fallen_idx]})")
            
            added = 0
            for split in ["train", "valid"]:
                src_imgs = FD1_DIR / split / "images"
                src_lbls = FD1_DIR / split / "labels"
                dst_imgs = DATASET_DIR / split / "images"
                dst_lbls = DATASET_DIR / split / "labels"
                
                if not src_imgs.exists():
                    continue
                
                for img_file in src_imgs.glob("*"):
                    lbl_file = src_lbls / (img_file.stem + ".txt")
                    if not lbl_file.exists():
                        continue
                    
                    # Copy image
                    dst_img = dst_imgs / f"fd1_{img_file.name}"
                    if not dst_img.exists():
                        shutil.copy(img_file, dst_img)
                    
                    # Remap label to 'fallen' class
                    dst_lbl = dst_lbls / f"fd1_{img_file.stem}.txt"
                    if not dst_lbl.exists():
                        lines = []
                        for line in lbl_file.read_text().strip().splitlines():
                            parts = line.split()
                            if len(parts) >= 5:
                                parts[0] = str(fallen_idx)  # Remap to fallen
                                lines.append(" ".join(parts))
                        if lines:
                            dst_lbl.write_text("\n".join(lines))
                            added += 1
            
            print(f"  Added {added} images from Fall-Detection-1")
        else:
            print("Could not find 'fallen' class in dataset")
    else:
        print(f"Fall-Detection-1 not found at {FD1_DIR}")
else:
    print("Skipping augmentation (not needed or Kaggle mode)")

In [ ]:
# Re-check class distribution after augmentation
if DATA_YAML.exists():
    lbl_dir = DATASET_DIR / "train" / "labels"
    if lbl_dir.exists():
        class_counts = Counter()
        for lbl_file in lbl_dir.glob("*.txt"):
            for line in lbl_file.read_text().strip().splitlines():
                if line.strip():
                    cls_id = int(line.split()[0])
                    class_counts[cls_id] += 1
        
        print(f"Final class distribution (train):")
        total = sum(class_counts.values())
        for cls_id in sorted(class_counts.keys()):
            name = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)
            pct = 100 * class_counts[cls_id] / total
            print(f"  [{cls_id}] {name}: {class_counts[cls_id]} ({pct:.1f}%)")
        print(f"  Total: {total} annotations")

## Section 4: Training Configuration

In [ ]:
# Update data.yaml paths for training
with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

data_cfg["path"] = str(DATASET_DIR)
data_cfg["train"] = "train/images"
data_cfg["val"] = "valid/images"

TRAIN_DATA_YAML = RESULTS_DIR / "data.yaml"
with open(TRAIN_DATA_YAML, "w") as f:
    yaml.dump(data_cfg, f)

print(f"Training data.yaml: {TRAIN_DATA_YAML}")
print(yaml.dump(data_cfg, default_flow_style=False))

In [ ]:
# Training configuration
import gc

def reset_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

reset_cuda()

model = YOLO("yolov8n.pt")

# Training config (same as V2 but with more epochs)
TRAIN_CONFIG = {
    "data": str(TRAIN_DATA_YAML),
    "epochs": 100,
    "imgsz": 640,
    "batch": 16,
    "patience": 30,
    "project": str(RESULTS_DIR / "runs"),
    "name": "v4_multiclass",
    "exist_ok": True,
    "verbose": True,
    "cache": True,
    "device": DEVICE,
    
    # Industrial augmentation
    "augment": True,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "degrees": 15,
    "translate": 0.1,
    "scale": 0.5,
    "mosaic": 1.0,
    "mixup": 0.1,
    "erasing": 0.4,  # Simulate occlusion
}

print("Training configuration:")
for k, v in TRAIN_CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Train!
print("Starting training...")
results = model.train(**TRAIN_CONFIG)
print("Training complete!")

## Section 5: Load Trained Model

In [ ]:
# Find best checkpoint
import glob

BEST_PT = None
search_patterns = [
    RESULTS_DIR / "runs" / "v4_multiclass" / "weights" / "best.pt",
    RESULTS_DIR / "runs" / "v4_multiclass*" / "weights" / "best.pt",
]

for pattern in search_patterns:
    if "*" in str(pattern):
        matches = glob.glob(str(pattern))
        if matches:
            BEST_PT = Path(sorted(matches)[-1])
            break
    elif Path(pattern).exists():
        BEST_PT = Path(pattern)
        break

if BEST_PT and BEST_PT.exists():
    print(f"Best checkpoint: {BEST_PT}")
    print(f"Size: {BEST_PT.stat().st_size / 1024 / 1024:.2f} MB")
    
    # Validate
    val_model = YOLO(str(BEST_PT))
    val_results = val_model.val(data=str(TRAIN_DATA_YAML), verbose=False)
    
    print(f"\nValidation Results:")
    print(f"  mAP50: {val_results.box.map50:.4f}")
    print(f"  mAP50-95: {val_results.box.map:.4f}")
    print(f"  Classes: {val_model.names}")
else:
    print("No checkpoint found - run training first")
    BEST_PT = None

## Section 6: MultiClassFallPipeline

Key difference from V2: **Require transition detection** (standing→falling→fallen)

In [ ]:
class MultiClassFallPipeline:
    """
    4-class detector with transition-based fall detection.
    
    Classes: standing, bending, falling, fallen
    
    Alert logic:
    - Track person with BoTSORT
    - Detect class sequence: normal (standing/bending) → falling → fallen
    - Confirm with temporal filter (N consecutive "fallen" frames)
    - Require transition: must see "falling" before "fallen" (reduces false alarms)
    """
    
    def __init__(self, model_path,
                 confirm_frames=15,       # Frames of "fallen" to confirm
                 cooldown_frames=150,     # 5 sec between alerts @ 30fps
                 require_transition=True, # Must see falling→fallen
                 min_confidence=0.4,
                 history_len=90):
        
        self.model = YOLO(str(model_path))
        self.confirm_frames = confirm_frames
        self.cooldown_frames = cooldown_frames
        self.require_transition = require_transition
        self.min_confidence = min_confidence
        self.history_len = history_len
        
        # Map class names to categories
        self._class_map = {}
        self.reset()
    
    def _get_category(self, class_name):
        """Map detected class to category: normal, falling, fallen"""
        name_lower = class_name.lower()
        if name_lower in ["standing", "bending"]:
            return "normal"
        elif name_lower == "falling":
            return "falling"
        elif name_lower in ["fallen", "fall-detected", "fall"]:
            return "fallen"
        return "unknown"
    
    def reset(self):
        """Reset state for new video."""
        self._history = defaultdict(lambda: deque(maxlen=self.history_len))
        self._saw_falling = defaultdict(bool)  # Track if we saw "falling" class
        self._alerted = defaultdict(bool)
        self._last_alert_frame = defaultdict(lambda: -self.cooldown_frames)
        self._events = []
        self._frame_count = 0
        self.model.predictor = None
    
    def process_frame(self, frame, frame_idx=None):
        """
        Process single frame.
        
        Returns: (events, annotated_frame, debug_info)
        """
        if frame_idx is None:
            frame_idx = self._frame_count
        self._frame_count = frame_idx + 1
        
        annotated = frame.copy()
        debug_info = {"tracks": [], "detections": 0}
        events = []
        
        # Run detection + tracking
        results = self.model.track(
            frame,
            persist=True,
            tracker="botsort.yaml",
            conf=self.min_confidence,
            iou=0.45,
            verbose=False
        )
        
        if not results or results[0].boxes is None:
            return events, annotated, debug_info
        
        boxes = results[0].boxes
        names = results[0].names
        
        if boxes.id is None:
            return events, annotated, debug_info
        
        track_ids = boxes.id.cpu().numpy().astype(int)
        class_ids = boxes.cls.cpu().numpy().astype(int)
        confidences = boxes.conf.cpu().numpy()
        xyxy_boxes = boxes.xyxy.cpu().numpy()
        
        debug_info["detections"] = len(track_ids)
        
        for tid, cid, conf, bbox in zip(track_ids, class_ids, confidences, xyxy_boxes):
            class_name = names.get(cid, "unknown")
            category = self._get_category(class_name)
            
            # Update history
            self._history[tid].append(category)
            
            # Track if we've seen "falling" for this person
            if category == "falling":
                self._saw_falling[tid] = True
            elif category == "normal":
                self._saw_falling[tid] = False  # Reset if back to normal
            
            debug_info["tracks"].append({
                "id": int(tid),
                "class": class_name,
                "category": category,
                "conf": float(conf),
                "saw_falling": self._saw_falling[tid]
            })
            
            hist = list(self._history[tid])
            
            # Check recovery (back to normal)
            if self._alerted[tid]:
                normal_count = sum(1 for h in hist[-30:] if h == "normal")
                if normal_count >= 20:  # 20/30 frames normal = recovered
                    self._alerted[tid] = False
                    self._saw_falling[tid] = False
                
                # Draw red for alerted
                x1, y1, x2, y2 = [int(c) for c in bbox]
                cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 0, 255), 3)
                cv2.putText(annotated, f"FALL! ID:{tid}", (x1, y1 - 10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                continue
            
            # Check for new fall
            fallen_tail = hist[-self.confirm_frames:]
            consecutive_fallen = (
                len(fallen_tail) == self.confirm_frames and
                all(h == "fallen" for h in fallen_tail)
            )
            
            # Must have been normal before
            had_normal = any(h == "normal" for h in hist[:-self.confirm_frames])
            
            # Must have seen "falling" transition (if required)
            transition_ok = (not self.require_transition) or self._saw_falling[tid]
            
            # Check cooldown
            frames_since_alert = frame_idx - self._last_alert_frame[tid]
            cooldown_ok = frames_since_alert >= self.cooldown_frames
            
            if consecutive_fallen and had_normal and transition_ok and cooldown_ok:
                self._alerted[tid] = True
                self._last_alert_frame[tid] = frame_idx
                
                ev = {
                    "frame_idx": frame_idx,
                    "track_id": int(tid),
                    "event_type": "fall_confirmed",
                    "confidence": float(conf),
                    "bbox": bbox.tolist(),
                    "timestamp": frame_idx / 30.0,
                    "had_transition": self._saw_falling[tid]
                }
                events.append(ev)
                self._events.append(ev)
            
            # Draw bounding box
            x1, y1, x2, y2 = [int(c) for c in bbox]
            if category == "fallen":
                color = (0, 255, 255)  # Yellow
            elif category == "falling":
                color = (0, 165, 255)  # Orange
            else:
                color = (0, 255, 0)    # Green
            
            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
            cv2.putText(annotated, f"{tid}:{class_name[:4]} {conf:.2f}", (x1, y1 - 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        
        return events, annotated, debug_info
    
    def run_on_video(self, video_path, return_raw_history=False):
        """Process entire video."""
        self.reset()
        cap = cv2.VideoCapture(str(video_path))
        all_events = []
        raw_history = defaultdict(list)
        
        frame_idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            events, _, debug = self.process_frame(frame, frame_idx)
            all_events.extend(events)
            
            if return_raw_history:
                for track in debug["tracks"]:
                    raw_history[track["id"]].append((frame_idx, track["category"]))
            
            frame_idx += 1
        
        cap.release()
        
        if return_raw_history:
            return all_events, dict(raw_history)
        return all_events


print("MultiClassFallPipeline defined.")
print("Key feature: Transition detection (must see 'falling' before 'fallen')")

## Section 7: Validation on 86 Videos

In [ ]:
# Load validation manifest
MANIFEST_PATH = PROJECT_DIR / "shared_val_manifest.json"

if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH) as f:
        manifest = json.load(f)
    
    gt_lookup = {v["video_uid"]: v for v in manifest["videos"]}
    
    print(f"Validation set: {len(manifest['videos'])} videos")
    print(f"  Fall videos: {sum(1 for v in manifest['videos'] if v['has_fall'])}")
    print(f"  ADL videos: {sum(1 for v in manifest['videos'] if not v['has_fall'])}")
else:
    print(f"Manifest not found: {MANIFEST_PATH}")
    manifest = None

In [ ]:
# Helper to resolve video paths
def resolve_video_path(video_path):
    p = Path(video_path)
    if p.exists():
        return str(p)
    try:
        rel = p.relative_to(PROJECT_DIR)
        extras_path = EXTRAS_DIR / rel
        if extras_path.exists():
            return str(extras_path)
    except ValueError:
        pass
    return None

In [ ]:
# Run validation
if manifest and BEST_PT and BEST_PT.exists():
    # Test with and without transition requirement
    for require_trans in [True, False]:
        print(f"\n{'='*60}")
        print(f"Testing with require_transition={require_trans}")
        print(f"{'='*60}")
        
        pipeline = MultiClassFallPipeline(
            BEST_PT,
            confirm_frames=10,
            require_transition=require_trans
        )
        
        results = {}
        
        for i, vid in enumerate(manifest["videos"]):
            video_path = resolve_video_path(vid["video_path"])
            if video_path is None:
                continue
            
            events = pipeline.run_on_video(video_path)
            results[vid["video_uid"]] = {
                "fall_detected": len(events) > 0,
                "events": events,
                "num_events": len(events)
            }
            
            if (i + 1) % 20 == 0:
                print(f"  Processed {i+1}/{len(manifest['videos'])} videos")
        
        # Compute metrics
        detected = 0
        missed = 0
        fa_vids = 0
        total_fas = 0
        delays = []
        
        for video_uid, res in results.items():
            gt = gt_lookup.get(video_uid, {})
            has_fall = gt.get("has_fall", False)
            fall_start = gt.get("fall_start_frame", 0)
            
            if has_fall:
                if res["fall_detected"]:
                    detected += 1
                    if res["events"]:
                        delay = res["events"][0]["frame_idx"] - fall_start
                        delays.append(delay)
                else:
                    missed += 1
            else:
                if res["fall_detected"]:
                    fa_vids += 1
                    total_fas += res["num_events"]
        
        total_fall = detected + missed
        med_delay = np.median(delays) if delays else float("nan")
        
        print(f"\nResults (require_transition={require_trans}):")
        print(f"  Detection: {detected}/{total_fall} ({100*detected/total_fall:.1f}%)")
        print(f"  Missed: {missed}")
        print(f"  FA videos: {fa_vids}")
        print(f"  Total FAs: {total_fas}")
        print(f"  Median delay: {med_delay:.1f} frames")
        
        # Save results
        suffix = "trans" if require_trans else "notrans"
        with open(RESULTS_DIR / f"v4_results_{suffix}.json", "w") as f:
            json.dump(results, f, indent=2)
else:
    print("Skip validation (no manifest or model)")

## Section 8: Comparison Table

In [ ]:
# Comparison with V2 and V3
print("="*70)
print("PIPELINE COMPARISON")
print("="*70)
print(f"{'Pipeline':<30} {'Detected':<12} {'FA Total':<10} {'Speed':<10}")
print("-"*70)
print(f"{'V3 Pose-CNN':<30} {'53/55 (96%)':<12} {'57':<10} {'Slow':<10}")
print(f"{'V2 Multi-class':<30} {'44/55 (80%)':<12} {'26':<10} {'25 FPS':<10}")
print(f"{'V3 YOLOv8n (2-class)':<30} {'15/55 (27%)':<12} {'1':<10} {'Fast':<10} ← FAILED")
if manifest and BEST_PT:
    print(f"{'V4 Multi-class (this)':<30} {f'{detected}/{total_fall} ({100*detected/total_fall:.0f}%)':<12} {f'{total_fas}':<10} {'~25 FPS':<10}")
print("="*70)

## Section 9: Export

In [ ]:
# Save model and config
if BEST_PT and BEST_PT.exists():
    # Copy model
    output_model = RESULTS_DIR / "v4_multiclass_best.pt"
    shutil.copy(BEST_PT, output_model)
    print(f"Model: {output_model}")
    
    # Save config
    config = {
        "model": "yolov8n",
        "classes": CLASS_NAMES,
        "confirm_frames": 10,
        "require_transition": True,
        "cooldown_frames": 150,
    }
    with open(RESULTS_DIR / "config.json", "w") as f:
        json.dump(config, f, indent=2)
    print(f"Config: {RESULTS_DIR / 'config.json'}")

print(f"\nAll results in: {RESULTS_DIR}")

In [ ]:
# Final summary
print("="*70)
print("V4 MULTI-CLASS FALL DETECTION - SUMMARY")
print("="*70)
print()
print("ARCHITECTURE:")
print("  Single YOLOv8n with 4 classes: standing, bending, falling, fallen")
print("  Transition detection: must see 'falling' before 'fallen'")
print("  BoTSORT tracking + temporal filtering")
print()
print("WHY THIS WORKS:")
print("  1. Multi-class captures TRANSITION (falling = action)")
print("  2. V3 YOLOv8n (2-class) failed because it only saw static state")
print("  3. Transition requirement reduces false alarms")
print()
print("TARGET vs V2:")
print("  V2: 80% detection, 26 false alerts")
print("  V4: >90% detection, <20 false alerts (with more training data)")
print("="*70)